# Character CTC → fine-tuned Qwen3 translation evaluation

이 노트북은 **추가 학습 없이** 기존 character CTC를 평가합니다.

```text
cached Kimi audio features
→ frozen 2-layer BiGRU character CTC
→ valid Myanmar Unicode text
→ fine-tuned Qwen3 translator
→ English / Korean translation
```

Qwen-token CTC 실험과 정확히 같은 held-out 300개를 사용합니다.

- English adapter: `clean_lora_best`
- Korean adapter: `korean_clean_lora_epoch1`
- 비교 기준: Qwen-token CTC greedy 및 fine-tuned Oracle
- test set과 Qwen3-TTS는 사용하지 않습니다.


In [ ]:
!pip -q uninstall -y torchao
!pip -q install -U "transformers==4.57.3" "peft==0.17.1" accelerate safetensors pandas sacrebleu jiwer


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc, re, unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sacrebleu.metrics import CHRF, BLEU
from jiwer import cer

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
ROOT=Path('/content/drive/MyDrive/S2ST')
SEM_DIR=ROOT/'baselines/bigru_prefix_semantic_alignment'
QWEN_CTC_DIR=ROOT/'baselines/qwen_token_ctc_nbest'
LORA_ROOT=ROOT/'baselines/qwen3_translation_lora'
EN_DIR=LORA_ROOT/'clean_lora_best'
KO_DIR=LORA_ROOT/'korean_clean_lora_epoch1'
OUT_DIR=LORA_ROOT/'character_ctc_eval'
OUT_DIR.mkdir(parents=True,exist_ok=True)

MANIFEST_PATH=SEM_DIR/'manifest.csv'
FEATURE_DIR=SEM_DIR/'kimi_features'
CTC_PATH=SEM_DIR/'bigru_ctc_best.pt'
FIXED_EVAL_PATH=QWEN_CTC_DIR/'dev_nbest.pkl'
LLM_ID='Qwen/Qwen3-4B-Instruct-2507'
TRANSLATION_BATCH=8
MAX_NEW_TOKENS=128

for p in [MANIFEST_PATH,CTC_PATH,FIXED_EVAL_PATH,EN_DIR,KO_DIR]: assert p.exists(),f'없음: {p}'
assert torch.cuda.is_available(),'GPU runtime을 선택하세요.'
print(torch.cuda.get_device_name(),DTYPE,OUT_DIR)


Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


NVIDIA RTX PRO 6000 Blackwell Server Edition torch.bfloat16 /content/drive/MyDrive/S2ST/baselines/qwen3_translation_lora/character_ctc_eval


## 1. 동일한 held-out 300개와 character vocabulary 복원


In [ ]:
def clean_text(x):
    if x is None or (isinstance(x,float) and np.isnan(x)): return ''
    return ' '.join(unicodedata.normalize('NFC',str(x)).strip().split())

manifest=pd.read_csv(MANIFEST_PATH,dtype={'sample_id':str})
for c in ['transcription_my','translation_en','translation_ko']:
    manifest[c]=manifest[c].map(clean_text)
assert len(manifest)==8897 and not manifest.sample_id.duplicated().any()

# Notebook 13/16과 완전히 같은 sample IDs를 사용합니다.
fixed=pd.read_pickle(FIXED_EVAL_PATH)
assert len(fixed)==300 and fixed.sample_id.astype(str).nunique()==300
fixed_ids=fixed.sample_id.astype(str).tolist()
selected=(manifest.set_index('sample_id').loc[fixed_ids].reset_index())
assert selected.sample_id.tolist()==fixed_ids

# CTC 학습 당시와 같은 전체 manifest, 같은 정렬 규칙입니다.
characters=sorted(set(''.join(manifest.transcription_my.tolist())))
char_to_id={'<blank>':0,'<unk>':1}
for ch in characters:
    if ch not in char_to_id: char_to_id[ch]=len(char_to_id)
id_to_char={v:k for k,v in char_to_id.items()}
BLANK_ID=char_to_id['<blank>'];UNK_ID=char_to_id['<unk>']

ckpt=torch.load(CTC_PATH,map_location='cpu',weights_only=True)
state=ckpt.get('model',ckpt)
checkpoint_vocab=int(state['head.2.weight'].shape[0])
assert len(char_to_id)==checkpoint_vocab,(
    f'문자 사전 불일치: manifest={len(char_to_id)}, checkpoint={checkpoint_vocab}. '
    'bigru_prefix_semantic_alignment 노트북이 만든 원래 manifest.csv가 필요합니다.'
)
print('samples:',len(selected),'sources:',selected.source.value_counts().to_dict(),
      'vocab:',len(char_to_id),'checkpoint epoch:',ckpt.get('epoch'))


samples: 300 sources: {'fleurs': 100, 'mig-burmese-audio-transcription': 100, 'openslr': 100} vocab: 166 checkpoint epoch: 12


## 2. 선택된 Kimi features만 로드하고 character CTC 실행


In [ ]:
needed=set(fixed_ids); feature_by_id={}
# Google Drive가 만든 `features-00003 (1).pt` 같은 중복 사본은 무시합니다.
feature_files=[FEATURE_DIR/f'features-{i:05d}.pt' for i in range(5)]
missing_feature_files=[p for p in feature_files if not p.exists()]
assert not missing_feature_files,f'정규 feature shard 누락: {missing_feature_files}'
print('using canonical feature shards:',[p.name for p in feature_files])
for path in tqdm(feature_files,desc='load selected Kimi features'):
    obj=torch.load(path,map_location='cpu',weights_only=False)
    for sid,feat in zip(map(str,obj['sample_ids']),obj['features']):
        if sid in needed: feature_by_id[sid]=feat.float()
    del obj;gc.collect()
missing=needed-set(feature_by_id)
assert not missing,f'feature 누락: {list(missing)[:5]}'

class BiGRUAdapter(nn.Module):
    def __init__(self,input_dim=1280,projection_dim=256,hidden_dim=256,layers=2,dropout=.1):
        super().__init__()
        self.input=nn.Sequential(nn.LayerNorm(input_dim),nn.Linear(input_dim,projection_dim),nn.GELU(),nn.Dropout(dropout))
        self.gru=nn.GRU(projection_dim,hidden_dim,num_layers=layers,batch_first=True,bidirectional=True,
                        dropout=dropout if layers>1 else 0)
    def forward(self,x,lengths):
        x=self.input(x)
        packed=pack_padded_sequence(x,lengths.cpu(),batch_first=True,enforce_sorted=False)
        packed,_=self.gru(packed)
        x,_=pad_packed_sequence(packed,batch_first=True,total_length=x.shape[1])
        return x

class CTCModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.adapter=BiGRUAdapter()
        self.head=nn.Sequential(nn.LayerNorm(512),nn.Dropout(.1),nn.Linear(512,len(char_to_id)))
    def forward(self,x,lengths): return self.head(self.adapter(x,lengths))

class FeatureDataset(Dataset):
    def __len__(self): return len(selected)
    def __getitem__(self,i):
        sid=str(selected.iloc[i].sample_id)
        return sid,feature_by_id[sid]

def collate(batch):
    ids=[x[0] for x in batch];feats=[x[1].float() for x in batch]
    lengths=torch.tensor([len(x) for x in feats],dtype=torch.long)
    return ids,pad_sequence(feats,batch_first=True),lengths

def collapse_decode(token_ids):
    chars=[];previous=None
    for token in token_ids:
        token=int(token)
        if token!=previous and token not in (BLANK_ID,UNK_ID): chars.append(id_to_char[token])
        previous=token
    return clean_text(''.join(chars))

# float32로 실행하여 이전 Half/Float layer_norm 충돌을 피합니다.
ctc=CTCModel().to(DEVICE,dtype=torch.float32)
ctc.load_state_dict(state);ctc.eval()
loader=DataLoader(FeatureDataset(),batch_size=16,shuffle=False,num_workers=0,collate_fn=collate)
pred_by_id={}
with torch.inference_mode():
    for ids,features,lengths in tqdm(loader,desc='character CTC greedy'):
        logits=ctc(features.to(DEVICE,dtype=torch.float32),lengths)
        pred=logits.argmax(-1).cpu()
        for sid,tokens,n in zip(ids,pred,lengths): pred_by_id[sid]=collapse_decode(tokens[:int(n)])

selected['character_ctc_text']=selected.sample_id.map(pred_by_id)
selected['sample_cer']=selected.apply(lambda r:cer(r.transcription_my,r.character_ctc_text),axis=1)
selected.to_csv(OUT_DIR/'character_ctc_transcriptions.csv',index=False)

def cer_row(name,g):
    return {'source':name,'samples':len(g),
            'cer_with_spaces':cer(' '.join(g.transcription_my),' '.join(g.character_ctc_text)),
            'cer_no_spaces':cer(''.join(g.transcription_my.str.replace(' ', '',regex=False)),
                                ''.join(g.character_ctc_text.str.replace(' ', '',regex=False))),
            'mean_sample_cer':g.sample_cer.mean(),
            'replacement_char_rate':g.character_ctc_text.str.contains('�',regex=False).mean()}
ctc_metrics=[cer_row('all',selected)]+[cer_row(s,g) for s,g in selected.groupby('source')]
ctc_metrics=pd.DataFrame(ctc_metrics)
display(ctc_metrics)
display(selected[['sample_id','source','transcription_my','character_ctc_text','sample_cer']].head(12))
ctc_metrics.to_csv(OUT_DIR/'character_ctc_metrics.csv',index=False)

del ctc,state,feature_by_id,loader
gc.collect();torch.cuda.empty_cache()


using canonical feature shards: ['features-00000.pt', 'features-00001.pt', 'features-00002.pt', 'features-00003.pt', 'features-00004.pt']


load selected Kimi features:   0%|          | 0/5 [00:00<?, ?it/s]

character CTC greedy:   0%|          | 0/19 [00:00<?, ?it/s]

,source,samples,cer_with_spaces,cer_no_spaces,mean_sample_cer,replacement_char_rate
0,all,300,0.164332,0.155071,0.174476,0.0
1,fleurs,100,0.164885,0.155819,0.166489,0.0
2,mig-burmese-audio-transcription,100,0.187381,0.185118,0.207981,0.0
3,openslr,100,0.149834,0.135152,0.148958,0.0


,sample_id,source,transcription_my,character_ctc_text,sample_cer
0,train_001180,fleurs,အစိုးရစုံစမ်းစစ်ဆေးသူများသည် ဗုဒ္ဓဟူးနေ့၏ ပျံသ...,အစိုးရစုံစမ်းစစ်ဆေးသူများသည် ဗုဒ်တဟူနေစ့်၏ ပြန...,0.188312
1,train_002169,fleurs,ပုံမှန် မီးငြှိမ်းသတ်ရေးဆိုင်ရာညွှန်ကြားမှုစနစ...,ပုံမန်မိးငြိမ်းသက်ရေးဆိုင်ရာ ညွန်ကျေားမှုစနစ်စ...,0.159817
2,train_001500,fleurs,“အဖျားနှင့် လည်ချောင်းနာသည်လွဲ၍ နေအိမ်မှ ဆက်သွ...,အဖျားနှင့် လည်ကြောင်းနာသည် လွယ်၍ နေမှဆက်သွယ် လ...,0.164706
3,train_000775,fleurs,ဒါကတော့ eBay ၏ သမိုင်းကြောင်းတွင် အကြီးမားဆုံး...,ဒါကတော့ အီပေ၏ သမိုင်းကြောင်းလည်အကြီးမားဆုံးသော...,0.147059
4,train_002146,fleurs,နာမည်ကြီး ဂရိလူမျိုးရှေ့နေများဖြစ်သော စကစ်ကက်ခ...,နာမည်ကြီးရိလူမျိုး ရှေ့နေများဖြစ်သော စကတ်ကခြာ ...,0.224576
5,train_003171,fleurs,လူ့ယဉ်ကျေးမှုတိုးတက်ခြင်းသည် အဖွဲ့အစည်းတစ်ခုအတ...,လုတ့်ယဉ်ကျေမှုသိုသတ်ချြင်သည် အဖွဲ့အစေ်တစ်ခုအတွ...,0.137405
6,train_000253,fleurs,"""အတိုချုံးပြောရလျှင် သင်၏ကားကိုအသုံးပြုခြင်းသည...",အတိုအချုံးပြောရလျှင် သင၏ခါကို အသုံးပြုခြင်းသည်...,0.125683
7,train_003852,fleurs,ကွင်းလယ်ရိုက်သူများဖြစ်သည့် စရှင်တန်အူဂါ နှင့်...,ကွင်လယ်ရိုက်သူများဖြစ်သည် ဆရှင်ဒန် ဥဂာနင့် ရဟူ...,0.172662
8,train_001589,fleurs,ခရီးသွားစာရေးခြင်းလုပ်ငန်းတွင် စတင်လုပ်ဆောင်ရန...,ခရီးသွားစာရေးချင်းလုပ်ငန်တွင် စတင်လုပ်စော်ရင်အ...,0.079755
9,train_001520,fleurs,ဆုပေးပွဲတွင် အတ္တလန်နာကျောင်းများက ရမှတ်များစစ...,ဆုပေးပွဲတွင် အတ်တလန်တာကျောင်းများက ရာမတ်များစစ...,0.127451


## 3. Fine-tuned English/Korean Qwen3 adapters 로드


In [ ]:
tokenizer=AutoTokenizer.from_pretrained(EN_DIR,use_fast=True)
if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
base=AutoModelForCausalLM.from_pretrained(LLM_ID,dtype=DTYPE,device_map={'':'cuda:0'},attn_implementation='sdpa')
model=PeftModel.from_pretrained(base,EN_DIR,adapter_name='english',is_trainable=False)
model.load_adapter(KO_DIR,adapter_name='korean',is_trainable=False)
model.eval()
print('loaded:',list(model.peft_config))


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

loaded: ['english', 'korean']


## 4. Character CTC text를 번역


In [ ]:
def make_prompt(text,lang):
    target='English' if lang=='en' else 'Korean'
    return f'Translate the following Myanmar text into {target}. Preserve its meaning and return only the translation.\n\nMyanmar:\n{text}'

@torch.inference_mode()
def generate_batch(texts,lang):
    model.set_adapter('english' if lang=='en' else 'korean')
    prompts=[tokenizer.apply_chat_template([{'role':'user','content':make_prompt(x,lang)}],
              tokenize=False,add_generation_prompt=True) for x in texts]
    inp=tokenizer(prompts,return_tensors='pt',padding=True,truncation=True,max_length=768).to(DEVICE)
    out=model.generate(**inp,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,repetition_penalty=1.05,
                       eos_token_id=tokenizer.eos_token_id,pad_token_id=tokenizer.pad_token_id)
    new=out[:,inp.input_ids.shape[1]:]
    return [x.strip() for x in tokenizer.batch_decode(new,skip_special_tokens=True)]

rows=[]
for lang in ['en','ko']:
    predictions=[]
    for start in tqdm(range(0,len(selected),TRANSLATION_BATCH),desc=f'character CTC → {lang}'):
        texts=selected.character_ctc_text.iloc[start:start+TRANSLATION_BATCH].tolist()
        predictions.extend(generate_batch(texts,lang))
    for r,pred in zip(selected.itertuples(),predictions):
        rows.append({'sample_id':str(r.sample_id),'source':str(r.source),'language':lang,
                     'reference':getattr(r,f'translation_{lang}'),'my_text':r.transcription_my,
                     'character_ctc_text':r.character_ctc_text,'sample_cer':r.sample_cer,
                     'character_ctc_translation':pred})
pred_df=pd.DataFrame(rows)
pred_df.to_csv(OUT_DIR/'character_ctc_translation_predictions.csv',index=False)
print('saved:',len(pred_df))


character CTC → en:   0%|          | 0/38 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializin

character CTC → ko:   0%|          | 0/38 [00:00<?, ?it/s]

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='le

saved: 600


## 5. 번역 점수와 기존 Qwen-token CTC / Oracle 비교


In [ ]:
chrf=CHRF(word_order=2);bleu=BLEU(effective_order=True)
def score(g):
    refs=g.reference.tolist();pred=g.character_ctc_translation.tolist()
    return {'samples':len(g),'chrf_pp':chrf.corpus_score(pred,[refs]).score,
            'bleu':bleu.corpus_score(pred,[refs]).score,
            'empty_rate':float(np.mean([not x.strip() for x in pred]))}

metrics=[{'source':'all','language':'all',**score(pred_df)}]
metrics += [{'source':'all','language':lang,**score(g)} for lang,g in pred_df.groupby('language')]
metrics += [{'source':source,'language':lang,**score(g)}
            for (source,lang),g in pred_df.groupby(['source','language'])]
metric_df=pd.DataFrame(metrics)
display(metric_df)
metric_df.to_csv(OUT_DIR/'character_ctc_translation_metrics.csv',index=False)

# 사용자가 방금 얻은 Qwen-token CTC 점수 및 동일 LoRA의 Oracle 점수와 비교합니다.
comparison=[]
known={
    'en':{'qwen_token_ctc':25.152121,'oracle':44.980627},
    'ko':{'qwen_token_ctc':10.441507,'oracle':22.797998},
}
for lang in ['en','ko']:
    current=float(metric_df.query("source=='all' and language==@lang").iloc[0].chrf_pp)
    comparison += [
        {'language':lang,'condition':'Qwen-token CTC greedy','chrf_pp':known[lang]['qwen_token_ctc']},
        {'language':lang,'condition':'Character CTC greedy','chrf_pp':current},
        {'language':lang,'condition':'Fine-tuned Oracle','chrf_pp':known[lang]['oracle']},
    ]
comparison=pd.DataFrame(comparison)
display(comparison)

decision=[]
for lang in ['en','ko']:
    x=comparison[comparison.language.eq(lang)].set_index('condition').chrf_pp
    gain=x['Character CTC greedy']-x['Qwen-token CTC greedy']
    retained=x['Character CTC greedy']/x['Fine-tuned Oracle']
    decision.append({'language':lang,'character_vs_qwen_token_gain':gain,
                     'oracle_score_retained':retained,
                     'recommended_path':'character CTC' if gain>0 else 'revisit ASR'})
decision=pd.DataFrame(decision)
display(decision)
comparison.to_csv(OUT_DIR/'comparison.csv',index=False)
decision.to_csv(OUT_DIR/'decision.csv',index=False)

display(pred_df.groupby(['source','language'],group_keys=False).head(3)[
    ['sample_id','source','language','reference','my_text','character_ctc_text','character_ctc_translation']])


,source,language,samples,chrf_pp,bleu,empty_rate
0,all,all,600,15.226536,4.643590,0.0
1,all,en,300,17.118877,5.587951,0.0
2,all,ko,300,11.233340,2.866216,0.0
3,fleurs,en,100,16.877884,4.682185,0.0
4,fleurs,ko,100,11.576887,2.185933,0.0
5,mig-burmese-audio-transcription,en,100,16.991014,9.311221,0.0
6,mig-burmese-audio-transcription,ko,100,10.125788,4.778480,0.0
7,openslr,en,100,17.607481,3.984071,0.0
8,openslr,ko,100,11.319298,2.391126,0.0


,language,condition,chrf_pp
0,en,Qwen-token CTC greedy,25.152121
1,en,Character CTC greedy,17.118877
2,en,Fine-tuned Oracle,44.980627
3,ko,Qwen-token CTC greedy,10.441507
4,ko,Character CTC greedy,11.233340
5,ko,Fine-tuned Oracle,22.797998


,language,character_vs_qwen_token_gain,oracle_score_retained,recommended_path
0,en,-8.033244,0.380583,revisit ASR
1,ko,0.791833,0.492734,character CTC


,sample_id,source,language,reference,my_text,character_ctc_text,character_ctc_translation
0,train_001180,fleurs,en,Xinhua reported that government investigators ...,အစိုးရစုံစမ်းစစ်ဆေးသူများသည် ဗုဒ္ဓဟူးနေ့၏ ပျံသ...,အစိုးရစုံစမ်းစစ်ဆေးသူများသည် ဗုဒ်တဟူနေစ့်၏ ပြန...,Regulators were able to retrieve two black box...
1,train_002169,fleurs,en,A power failure following a routine fire-comma...,ပုံမှန် မီးငြှိမ်းသတ်ရေးဆိုင်ရာညွှန်ကြားမှုစနစ...,ပုံမန်မိးငြိမ်းသက်ရေးဆိုင်ရာ ညွန်ကျေားမှုစနစ်စ...,งูมังกรน้ำท่วมที่มีระบบการฝึกอบรมนี้ก็พุ่งออกไ...
2,train_001500,fleurs,en,"""Apart from the fever and a sore throat, I fee...",“အဖျားနှင့် လည်ချောင်းနာသည်လွဲ၍ နေအိမ်မှ ဆက်သွ...,အဖျားနှင့် လည်ကြောင်းနာသည် လွယ်၍ နေမှဆက်သွယ် လ...,"သည်는 열과 기침이 쉽게 통증을 일으키며, 상당히 편안한 침대에 앉아서 연락할 수..."
100,train_006319,mig-burmese-audio-transcription,en,Calculating that they could make us dance to t...,ငါတို့ စိတ်တိုင်းကျ ကြိုးဆွဲရာ ကခိုင်းလို့ ရမယ...,ဒါတို့ စိတ်တိုမ်းအရာ ကြိုဆွဲရာကခိန်ကော့ အရာမယ်...,"ပ악, ที่เป็นสิ่งที่ดึงดูดความคิดและทำให้รู้สึกม..."
101,train_006907,mig-burmese-audio-transcription,en,But in reality...,ဒါပေမဲ့ တကယ်မှာကကျတော့,ဒါပေမဲ့ တကယ်မှာကျတော့,Urged by the real situation...
102,train_006676,mig-burmese-audio-transcription,en,It was something they suddenly conceived and o...,အင်္ဂလန်က ပြန်ခါနီးမှာ ဘုရင်မကို နှုတ်ဆက်ဖို့ ...,အင်္ဂလန်က ပြန်ခနနီးမှာ ဘုရင်မောငက် နှုစ်ဆက်ဖို...,"သ고무 마웅, who had been waiting for the Queen to ..."
200,train_005665,openslr,en,"When we attended university, there was a unive...",ကျနော်တို့ တက္ကသိုလ် ရောက်စဉ် မှာ တက္ကသိုလ် အန...,ကျနော်တို့ သက္ကတို် ရောက်စင်မှာ သက္ကတို အနုပညာ...,"100 years ago, there was a Khitaw Art Associat..."
201,train_006056,openslr,en,I do not want to move to the other side before...,ဒီဘက် မပြီးပြတ်သေးဘဲ နဲ့ ဟိုဘက် လည်း မသွားချင်ဘူး,ဒီဘက် မပြီးဖပျက်သေး ပဲ နဲ့ ဟိုပတ် လည်း မသွားချ...,"300 won, but I don't want to go there either."
202,train_005043,openslr,en,"Although I listened to other songs, I could no...",တခြား သီချင်းတွေ နားထောင်ခဲ့ ပေမဲ့ မြန်မာ သံစဉ...,တခြား သီချင်းတွေ နားထောင်ခဲ့ ပေမဲ့ မြန်မာ သံစင...,"ပ다웅, นั่นอาจไม่ใช่เพราะเพลงนั้นเลวร้าย..."
300,train_001180,fleurs,ko,신화(Xinhua)통신은 정부 조사관들이 비행기록 장치 '블랙박스' 2개를 수요일에...,အစိုးရစုံစမ်းစစ်ဆေးသူများသည် ဗုဒ္ဓဟူးနေ့၏ ပျံသ...,အစိုးရစုံစမ်းစစ်ဆေးသူများသည် ဗုဒ်တဟူနေစ့်၏ ပြန...,워크스가 희극의 귀환 여정을 기록한 블로그 두 개를 복구했다고 밝혔다.


## 판정 기준

- Character CTC가 EN `25.15`, KO `10.44`를 넘으면 Qwen-token CTC를 폐기합니다.
- `replacement_char_rate`는 반드시 0이어야 합니다.
- Character CTC와 Oracle의 차이가 작으면 noisy adaptation 없이 그대로 TTS에 연결합니다.
- 차이가 크더라도 먼저 CTC head 개선을 검토하고, 전체 noisy adaptation은 마지막 선택지로 남깁니다.


In [ ]:
# 기존 일반 prompt 결과 보존
baseline_path = OUT_DIR / "character_ctc_translation_predictions.csv"

if baseline_path.exists():
    baseline_df = pd.read_csv(
        baseline_path,
        dtype={"sample_id": str},
    )
else:
    baseline_df = pred_df.copy()

# Decoder-only batch generation 안전 설정
tokenizer.padding_side = "left"


def make_prompt(text, lang):
    target = "English" if lang == "en" else "Korean"

    return f"""The following is an automatic speech recognition result for a spoken Myanmar utterance.

The transcription may contain:
- minor spelling errors
- incorrect spacing
- misplaced Myanmar combining characters
- substitutions between phonetically similar Myanmar words

Recover the intended meaning conservatively from the surrounding context and translate it into {target}.

Important:
- Do not translate isolated misspelled words literally when the intended Myanmar expression is clear from context.
- Do not invent names, dates, numbers, places, or facts.
- Do not add explanations.
- Do not output the corrected Myanmar sentence.
- Write entirely in {target}.
- Return only the translation.

Myanmar ASR transcription:
{text}"""


@torch.inference_mode()
def generate_batch(texts, lang):
    adapter_name = "english" if lang == "en" else "korean"
    model.set_adapter(adapter_name)

    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": make_prompt(text, lang)}],
            tokenize=False,
            add_generation_prompt=True,
        )
        for text in texts
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=768,
    ).to(DEVICE)

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        repetition_penalty=1.05,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    generated = outputs[:, inputs.input_ids.shape[1]:]

    return [
        text.strip()
        for text in tokenizer.batch_decode(
            generated,
            skip_special_tokens=True,
        )
    ]


asr_aware_rows = []

for lang in ["en", "ko"]:
    predictions = []

    for start in tqdm(
        range(0, len(selected), TRANSLATION_BATCH),
        desc=f"ASR-aware character CTC → {lang}",
    ):
        batch_texts = selected.character_ctc_text.iloc[
            start:start + TRANSLATION_BATCH
        ].tolist()

        predictions.extend(
            generate_batch(batch_texts, lang)
        )

    for row, prediction in zip(
        selected.itertuples(),
        predictions,
    ):
        asr_aware_rows.append({
            "sample_id": str(row.sample_id),
            "source": str(row.source),
            "language": lang,
            "reference": getattr(
                row,
                f"translation_{lang}",
            ),
            "my_text": row.transcription_my,
            "character_ctc_text": row.character_ctc_text,
            "sample_cer": row.sample_cer,
            "asr_aware_translation": prediction,
        })

asr_aware_df = pd.DataFrame(asr_aware_rows)

assert len(asr_aware_df) == 600
assert not asr_aware_df.asr_aware_translation.isna().any()

asr_aware_df.to_csv(
    OUT_DIR / "character_ctc_asr_aware_predictions.csv",
    index=False,
)

print("saved:", len(asr_aware_df))
display(asr_aware_df.head(6))

ASR-aware character CTC → en:   0%|          | 0/38 [00:00<?, ?it/s]

ASR-aware character CTC → ko:   0%|          | 0/38 [00:00<?, ?it/s]

saved: 600


,sample_id,source,language,reference,my_text,character_ctc_text,sample_cer,asr_aware_translation
0,train_001180,fleurs,en,Xinhua reported that government investigators ...,အစိုးရစုံစမ်းစစ်ဆေးသူများသည် ဗုဒ္ဓဟူးနေ့၏ ပျံသ...,အစိုးရစုံစမ်းစစ်ဆေးသူများသည် ဗုဒ်တဟူနေစ့်၏ ပြန...,0.188312,The Royal Commission's investigators recovered...
1,train_002169,fleurs,en,A power failure following a routine fire-comma...,ပုံမှန် မီးငြှိမ်းသတ်ရေးဆိုင်ရာညွှန်ကြားမှုစနစ...,ပုံမန်မိးငြိမ်းသက်ရေးဆိုင်ရာ ညွန်ကျေားမှုစနစ်စ...,0.159817,An earthquake monitoring system test triggered...
2,train_001500,fleurs,en,"""Apart from the fever and a sore throat, I fee...",“အဖျားနှင့် လည်ချောင်းနာသည်လွဲ၍ နေအိမ်မှ ဆက်သွ...,အဖျားနှင့် လည်ကြောင်းနာသည် လွယ်၍ နေမှဆက်သွယ် လ...,0.164706,It is easy to communicate and work together de...
3,train_000775,fleurs,en,It is the biggest acquisition in eBay's history.,ဒါကတော့ eBay ၏ သမိုင်းကြောင်းတွင် အကြီးမားဆုံး...,ဒါကတော့ အီပေ၏ သမိုင်းကြောင်းလည်အကြီးမားဆုံးသော...,0.147059,This was the greatest achievement of Einstein'...
4,train_002146,fleurs,en,"The famous Greek lawyers, Sakis Kechagioglou a...",နာမည်ကြီး ဂရိလူမျိုးရှေ့နေများဖြစ်သော စကစ်ကက်ခ...,နာမည်ကြီးရိလူမျိုး ရှေ့နေများဖြစ်သော စကတ်ကခြာ ...,0.224576,"The two prominent lawyers, Satya Kaba and Giri..."
5,train_003171,fleurs,en,A civilization is a singular culture shared by...,လူ့ယဉ်ကျေးမှုတိုးတက်ခြင်းသည် အဖွဲ့အစည်းတစ်ခုအတ...,လုတ့်ယဉ်ကျေမှုသိုသတ်ချြင်သည် အဖွဲ့အစေ်တစ်ခုအတွ...,0.137405,Car sharing is a form of carpooling where seve...


In [ ]:
from sacrebleu.metrics import CHRF, BLEU

chrf = CHRF(word_order=2)
bleu = BLEU(effective_order=True)

# 기존 결과를 같은 키로 결합
old_column = (
    "character_ctc_translation"
    if "character_ctc_translation" in baseline_df.columns
    else "prediction"
)

comparison_df = asr_aware_df.merge(
    baseline_df[
        [
            "sample_id",
            "language",
            old_column,
        ]
    ].rename(
        columns={
            old_column: "standard_translation",
        }
    ),
    on=["sample_id", "language"],
    how="left",
    validate="one_to_one",
)

assert comparison_df.standard_translation.notna().all()


def score_predictions(group, column):
    references = group.reference.fillna("").tolist()
    predictions = group[column].fillna("").tolist()

    return {
        "samples": len(group),
        "chrf_pp": chrf.corpus_score(
            predictions,
            [references],
        ).score,
        "bleu": bleu.corpus_score(
            predictions,
            [references],
        ).score,
        "empty_rate": float(
            np.mean([
                not text.strip()
                for text in predictions
            ])
        ),
    }


metric_rows = []

conditions = [
    ("all", "all"),
    ("all", "en"),
    ("all", "ko"),
]

conditions += [
    (source, lang)
    for source in sorted(comparison_df.source.unique())
    for lang in ["en", "ko"]
]

for source, lang in conditions:
    group = comparison_df

    if source != "all":
        group = group[group.source.eq(source)]

    if lang != "all":
        group = group[group.language.eq(lang)]

    standard = score_predictions(
        group,
        "standard_translation",
    )

    aware = score_predictions(
        group,
        "asr_aware_translation",
    )

    metric_rows.append({
        "source": source,
        "language": lang,
        "samples": len(group),
        "standard_chrf_pp": standard["chrf_pp"],
        "asr_aware_chrf_pp": aware["chrf_pp"],
        "chrf_gain": (
            aware["chrf_pp"]
            - standard["chrf_pp"]
        ),
        "standard_bleu": standard["bleu"],
        "asr_aware_bleu": aware["bleu"],
        "bleu_gain": (
            aware["bleu"]
            - standard["bleu"]
        ),
        "asr_aware_empty_rate": aware["empty_rate"],
    })

asr_aware_metrics = pd.DataFrame(metric_rows)

display(asr_aware_metrics)

asr_aware_metrics.to_csv(
    OUT_DIR / "asr_aware_vs_standard_metrics.csv",
    index=False,
)

,source,language,samples,standard_chrf_pp,asr_aware_chrf_pp,chrf_gain,standard_bleu,asr_aware_bleu,bleu_gain,asr_aware_empty_rate
0,all,all,600,15.226536,21.870583,6.644047,4.643590,6.609695,1.966105,0.0
1,all,en,300,17.118877,26.609848,9.490971,5.587951,8.774743,3.186793,0.0
2,all,ko,300,11.233340,11.715379,0.482038,2.866216,2.933726,0.067510,0.0
3,fleurs,en,100,16.877884,28.281615,11.403730,4.682185,8.596677,3.914492,0.0
4,fleurs,ko,100,11.576887,11.557100,-0.019787,2.185933,2.144834,-0.041100,0.0
5,mig-burmese-audio-transcription,en,100,16.991014,22.906794,5.915780,9.311221,11.542682,2.231461,0.0
6,mig-burmese-audio-transcription,ko,100,10.125788,10.795380,0.669592,4.778480,4.883646,0.105166,0.0
7,openslr,en,100,17.607481,26.272080,8.664599,3.984071,6.879518,2.895447,0.0
8,openslr,ko,100,11.319298,12.606101,1.286803,2.391126,2.399722,0.008595,0.0


In [ ]:
# Myanmar, Thai, Hangul 문자 범위
MYANMAR_RE = re.compile(r"[\u1000-\u109F\uAA60-\uAA7F]")
THAI_RE = re.compile(r"[\u0E00-\u0E7F]")
HANGUL_RE = re.compile(r"[\uAC00-\uD7A3\u1100-\u11FF\u3130-\u318F]")


def has_wrong_script(text, lang):
    text = str(text)

    if lang == "en":
        # 영어 출력에는 미얀마어, 태국어, 한글이 없어야 함
        return bool(
            MYANMAR_RE.search(text)
            or THAI_RE.search(text)
            or HANGUL_RE.search(text)
        )

    # 한국어에는 미얀마어와 태국어가 없어야 함
    return bool(
        MYANMAR_RE.search(text)
        or THAI_RE.search(text)
    )


comparison_df["standard_wrong_script"] = comparison_df.apply(
    lambda row: has_wrong_script(
        row.standard_translation,
        row.language,
    ),
    axis=1,
)

comparison_df["asr_aware_wrong_script"] = comparison_df.apply(
    lambda row: has_wrong_script(
        row.asr_aware_translation,
        row.language,
    ),
    axis=1,
)

script_metrics = (
    comparison_df
    .groupby("language")
    .agg(
        samples=("sample_id", "size"),
        standard_wrong_script_rate=(
            "standard_wrong_script",
            "mean",
        ),
        asr_aware_wrong_script_rate=(
            "asr_aware_wrong_script",
            "mean",
        ),
    )
    .reset_index()
)

display(script_metrics)

script_metrics.to_csv(
    OUT_DIR / "asr_aware_script_metrics.csv",
    index=False,
)

# 실제 변화 예시
changed_examples = comparison_df[
    comparison_df.standard_translation.ne(
        comparison_df.asr_aware_translation
    )
][
    [
        "sample_id",
        "source",
        "language",
        "reference",
        "character_ctc_text",
        "standard_translation",
        "asr_aware_translation",
    ]
]

display(
    changed_examples
    .groupby(
        ["source", "language"],
        group_keys=False,
    )
    .head(3)
)

,language,samples,standard_wrong_script_rate,asr_aware_wrong_script_rate
0,en,300,0.443333,0.0
1,ko,300,0.016667,0.0


,sample_id,source,language,reference,character_ctc_text,standard_translation,asr_aware_translation
0,train_001180,fleurs,en,Xinhua reported that government investigators ...,အစိုးရစုံစမ်းစစ်ဆေးသူများသည် ဗုဒ်တဟူနေစ့်၏ ပြန...,Regulators were able to retrieve two black box...,The Royal Commission's investigators recovered...
1,train_002169,fleurs,en,A power failure following a routine fire-comma...,ပုံမန်မိးငြိမ်းသက်ရေးဆိုင်ရာ ညွန်ကျေားမှုစနစ်စ...,งูมังกรน้ำท่วมที่มีระบบการฝึกอบรมนี้ก็พุ่งออกไ...,An earthquake monitoring system test triggered...
2,train_001500,fleurs,en,"""Apart from the fever and a sore throat, I fee...",အဖျားနှင့် လည်ကြောင်းနာသည် လွယ်၍ နေမှဆက်သွယ် လ...,"သည်는 열과 기침이 쉽게 통증을 일으키며, 상당히 편안한 침대에 앉아서 연락할 수...",It is easy to communicate and work together de...
100,train_006319,mig-burmese-audio-transcription,en,Calculating that they could make us dance to t...,ဒါတို့ စိတ်တိုမ်းအရာ ကြိုဆွဲရာကခိန်ကော့ အရာမယ်...,"ပ악, ที่เป็นสิ่งที่ดึงดูดความคิดและทำให้รู้สึกม...",They calculated that these were objects of des...
101,train_006907,mig-burmese-audio-transcription,en,But in reality...,ဒါပေမဲ့ တကယ်မှာကျတော့,Urged by the real situation...,But in reality...
102,train_006676,mig-burmese-audio-transcription,en,It was something they suddenly conceived and o...,အင်္ဂလန်က ပြန်ခနနီးမှာ ဘုရင်မောငက် နှုစ်ဆက်ဖို...,"သ고무 마웅, who had been waiting for the Queen to ...",The Queen's four sons were arranged in success...
200,train_005665,openslr,en,"When we attended university, there was a unive...",ကျနော်တို့ သက္ကတို် ရောက်စင်မှာ သက္ကတို အနုပညာ...,"100 years ago, there was a Khitaw Art Associat...","When we arrived at the pagoda, there was an ar..."
201,train_006056,openslr,en,I do not want to move to the other side before...,ဒီဘက် မပြီးဖပျက်သေး ပဲ နဲ့ ဟိုပတ် လည်း မသွားချ...,"300 won, but I don't want to go there either.",I do not want to go back until this matter is ...
202,train_005043,openslr,en,"Although I listened to other songs, I could no...",တခြား သီချင်းတွေ နားထောင်ခဲ့ ပေမဲ့ မြန်မာ သံစင...,"ပ다웅, นั่นอาจไม่ใช่เพราะเพลงนั้นเลวร้าย...","I listened to other songs too, but they were n..."
300,train_001180,fleurs,ko,신화(Xinhua)통신은 정부 조사관들이 비행기록 장치 '블랙박스' 2개를 수요일에...,အစိုးရစုံစမ်းစစ်ဆေးသူများသည် ဗုဒ်တဟူနေစ့်၏ ပြန...,워크스가 희극의 귀환 여정을 기록한 블로그 두 개를 복구했다고 밝혔다.,신문에 따르면 정부 조사관들은 띤다 헤인 제트의 환승 여정을 기록한 두 종이를 회수했다.
